<a href="https://colab.research.google.com/github/sayja-yug/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayja-yug/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. Ranked Actions + Reason Codes

## Purpose

This section converts the validated Week-5/Week-6 evidence into a practical
content-review queue for **Lane 2 — Refresh / Content Opportunity Scoring**.

The purpose is not to automatically change or publish content. The purpose is
to prioritize which content items should receive human review first.

The queue is based on information that was available during the March 2026
decision window. The April 2026 outcome was used previously for model
validation, but it is not used to create the Week-7 action queue.

## Evidence carried forward

The Week-5 model compared Logistic Regression and Random Forest against the
Week-4 transparent baseline. Week-6 then re-evaluated the model using a
client-grouped split and audited the features for future-window leakage.

The validated Week-6 comparison showed:

- Week-4 baseline Average Precision: 0.5564
- Week-5 Logistic Regression Average Precision under grouped validation: 0.5754

Therefore, the model is treated as **decision-support evidence**, not as a
guarantee that a refresh will improve performance.

## Action design

Each ranked item will contain:

- a pseudonymized client/content identifier;
- the model score used for prioritization;
- a reason code explaining the main observable signal;
- a recommended human-review action;
- a rank in the review queue.

Reason codes are intended to make the ranking understandable rather than
hiding the reason behind a high score.

The proposed action categories are:

- `REFRESH_REVIEW` — investigate whether the page needs content improvement.
- `MONITOR` — keep the page under observation when the evidence is weaker.
- `HOLD` — do not prioritize an intervention from this model alone.

These actions are recommendations for human review, not automated production
decisions.

## Leakage rule

The action queue must not use:

- April impressions;
- April clicks;
- April average position;
- `future_decline`;
- `click_change_pct`;
- Week-4 `baseline_score`;
- Week-4 `reason_code`;
- Week-4 `action_label`

as model inputs for creating the Week-7 queue.

The Week-4 and Week-6 outputs are evidence and comparison artifacts, not new
features.

## Intended interpretation

A high score means that the validated model ranks the item as a higher
priority candidate for review. It does not mean that the page is definitely
wrong, that a refresh will cause recovery, or that the model has established
causality.

The final decision remains with a human reviewer.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 7 — SECTION 1
# RANKED ACTIONS + REASON CODES
# Lane 2: Refresh / Content Opportunity Scoring
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 75)
print("WEEK 7 — SECTION 1: RANKED ACTIONS + REASON CODES")
print("=" * 75)

# ------------------------------------------------------------
# 1. Locate the Week-5 ranking output
# ------------------------------------------------------------

candidate_paths = [
    "/content/w05_test_rankings.csv",
    "/content/work/outputs/w05_test_rankings.csv",
    "w05_test_rankings.csv",
    "work/outputs/w05_test_rankings.csv"
]

ranking_path = None

for path in candidate_paths:
    if os.path.exists(path):
        ranking_path = path
        break

if ranking_path is None:
    raise FileNotFoundError(
        """
Week-5 ranking file was not found.

Expected one of:
- /content/w05_test_rankings.csv
- /content/work/outputs/w05_test_rankings.csv

Upload/copy w05_test_rankings.csv into Colab and rerun this cell.
"""
    )

print(f"✅ Week-5 ranking file found:")
print(ranking_path)


# ------------------------------------------------------------
# 2. Load ranking data
# ------------------------------------------------------------

ranking_df = pd.read_csv(ranking_path)

print("\n" + "=" * 75)
print("WEEK-5 RANKING DATA LOADED")
print("=" * 75)

print("Shape:", ranking_df.shape)

print("\nColumns:")
print(list(ranking_df.columns))


# ------------------------------------------------------------
# 3. Check the identifiers
# ------------------------------------------------------------

required_ids = [
    "client_hash_id",
    "content_hash_id"
]

missing_ids = [
    col for col in required_ids
    if col not in ranking_df.columns
]

if missing_ids:
    raise ValueError(
        f"Required identifier columns are missing: {missing_ids}"
    )

print("\n" + "=" * 75)
print("IDENTIFIER CHECK")
print("=" * 75)

print("✅ client_hash_id available")
print("✅ content_hash_id available")

print(
    "\nThese identifiers are pseudonyms used for joining/grouping "
    "and are not treated as model features."
)


# ------------------------------------------------------------
# 4. Identify the validated model score
# ------------------------------------------------------------

possible_model_scores = [
    "rf_score",
    "logistic_score",
    "model_score"
]

available_scores = [
    col
    for col in possible_model_scores
    if col in ranking_df.columns
]

if len(available_scores) == 0:
    raise ValueError(
        """
No validated model score was found.

Expected one of:
rf_score
logistic_score
model_score
"""
    )

print("\n" + "=" * 75)
print("MODEL SCORE CHECK")
print("=" * 75)

print("Available model scores:", available_scores)


# ------------------------------------------------------------
# 5. Use the Random Forest score
# ------------------------------------------------------------
# Week-5 showed Random Forest as the strongest model in the
# original Week-5 ranking evaluation.
#
# We use its score for the action queue when available.

if "logistic_score" in ranking_df.columns:
    SCORE_COLUMN = "logistic_score"
elif "rf_score" in ranking_df.columns:
    SCORE_COLUMN = "rf_score"
else:
    SCORE_COLUMN = available_scores[0]

print("Selected ranking score:", SCORE_COLUMN)


# ------------------------------------------------------------
# 6. Remove invalid score rows
# ------------------------------------------------------------

queue_df = ranking_df.copy()

queue_df[SCORE_COLUMN] = pd.to_numeric(
    queue_df[SCORE_COLUMN],
    errors="coerce"
)

queue_df = queue_df.dropna(
    subset=[SCORE_COLUMN]
).copy()

queue_df = queue_df[
    np.isfinite(queue_df[SCORE_COLUMN])
].copy()

print("\nRows after valid-score filtering:", len(queue_df))


# ------------------------------------------------------------
# 7. Check for forbidden future information
# ------------------------------------------------------------

forbidden_columns = [
    "april_impressions",
    "april_clicks",
    "april_avg_position",
    "click_change_pct",
    "future_decline"
]

forbidden_present = [
    col
    for col in forbidden_columns
    if col in queue_df.columns
]

print("\n" + "=" * 75)
print("FUTURE-WINDOW CHECK")
print("=" * 75)

if forbidden_present:
    print(
        "⚠️ Future/evaluation columns exist in the source file:",
        forbidden_present
    )
    print(
        "They will NOT be used to construct the Week-7 action queue."
    )
else:
    print("✅ No future-window columns detected.")


# ------------------------------------------------------------
# 8. Check Week-4 decision outputs
# ------------------------------------------------------------

week4_outputs = [
    "baseline_score",
    "reason_code",
    "action_label"
]

week4_present = [
    col
    for col in week4_outputs
    if col in queue_df.columns
]

print("\n" + "=" * 75)
print("WEEK-4 OUTPUT CHECK")
print("=" * 75)

print("Week-4 output columns present:", week4_present)

print(
    "\nWeek-4 outputs are retained only as historical comparison "
    "context. They are not used as model features."
)


# ------------------------------------------------------------
# 9. Basic score diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("MODEL SCORE SUMMARY")
print("=" * 75)

print(
    queue_df[SCORE_COLUMN].describe()
)


# ------------------------------------------------------------
# 10. Create a clean ranking
# ------------------------------------------------------------

queue_df = queue_df.sort_values(
    SCORE_COLUMN,
    ascending=False
).reset_index(drop=True)

queue_df["rank"] = np.arange(
    1,
    len(queue_df) + 1
)

print("\n" + "=" * 75)
print("RANKING CREATED")
print("=" * 75)

print("Total ranked items:", len(queue_df))

print("\nTop 10 ranked items:")

display(
    queue_df[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            SCORE_COLUMN
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 11. Final status
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SECTION 1 — DATA SETUP COMPLETE")
print("=" * 75)

print("Ranking source :", ranking_path)
print("Ranking score  :", SCORE_COLUMN)
print("Rows ranked    :", len(queue_df))

print(
    "\n✅ Week-5 ranking evidence loaded."
)

print(
    "✅ Model score identified."
)

print(
    "✅ Future-window fields are not used as action inputs."
)

print(
    "✅ Ranking created for the Week-7 action playbook."
)

print(
    "\nNext:"
    "\nSECTION 1 — assign reason codes and human-review actions."
)

WEEK 7 — SECTION 1: RANKED ACTIONS + REASON CODES
✅ Week-5 ranking file found:
/content/w05_test_rankings.csv

WEEK-5 RANKING DATA LOADED
Shape: (106, 6)

Columns:
['client_hash_id', 'content_hash_id', 'future_decline', 'baseline_score', 'logistic_score', 'rf_score']

IDENTIFIER CHECK
✅ client_hash_id available
✅ content_hash_id available

These identifiers are pseudonyms used for joining/grouping and are not treated as model features.

MODEL SCORE CHECK
Available model scores: ['rf_score', 'logistic_score']
Selected ranking score: logistic_score

Rows after valid-score filtering: 106

FUTURE-WINDOW CHECK
⚠️ Future/evaluation columns exist in the source file: ['future_decline']
They will NOT be used to construct the Week-7 action queue.

WEEK-4 OUTPUT CHECK
Week-4 output columns present: ['baseline_score']

Week-4 outputs are retained only as historical comparison context. They are not used as model features.

MODEL SCORE SUMMARY
count    106.000000
mean       0.552400
std        0.061

,rank,client_hash_id,content_hash_id,logistic_score
0,1,client_08a6a72ff48e62c0,content_1fbc173c6b0a0512,0.611929
1,2,client_08a6a72ff48e62c0,content_67c22dec4f120ed8,0.608996
2,3,client_3f0ce4d44fe94f3d,content_57195309e404a304,0.604387
3,4,client_08a6a72ff48e62c0,content_78501978aafda9e4,0.600780
4,5,client_a80fca3f171ed1de,content_95e0cd5a6f9d7391,0.600746
5,6,client_0fa64a184f18a4a0,content_474c280cef704f95,0.599018
6,7,client_0fa64a184f18a4a0,content_1a96c5be2c90593d,0.597749
7,8,client_0fa64a184f18a4a0,content_50c98fc9745d64a3,0.595253
8,9,client_08a6a72ff48e62c0,content_281440c260f75191,0.595245
9,10,client_08a6a72ff48e62c0,content_5f2c5708acd38231,0.594797



SECTION 1 — DATA SETUP COMPLETE
Ranking source : /content/w05_test_rankings.csv
Ranking score  : logistic_score
Rows ranked    : 106

✅ Week-5 ranking evidence loaded.
✅ Model score identified.
✅ Future-window fields are not used as action inputs.
✅ Ranking created for the Week-7 action playbook.

Next:
SECTION 1 — assign reason codes and human-review actions.


# 2. Intended Use and Limits

## What this playbook is intended to do

This playbook is designed for **Lane 2 — Refresh / Content Opportunity
Scoring**.

Its purpose is to help a human reviewer decide **which content items should
be reviewed first**.

The model score is therefore treated as a prioritization signal.

A higher score means that the item is ranked as a higher-priority candidate
for review relative to other items in this queue.

It does NOT mean that:

- the page is definitely declining;
- the page definitely needs a refresh;
- refreshing the page will improve performance;
- the model has established a causal relationship;
- the recommendation should be executed automatically.

The project guidance defines Lane 2 as a ranked review queue with scores,
actions, and reason codes. The queue therefore supports prioritization rather
than autonomous content changes.

## What a human reviewer should use the queue for

A reviewer can use a high-ranked item as a starting point for investigation.

The reviewer should examine the observable evidence available during the
decision window, including:

- search impressions;
- search clicks;
- average search position;
- the model's ranking score;
- the reason code assigned by the playbook.

The reviewer should then decide whether the item actually warrants:

- a refresh review;
- further investigation;
- monitoring;
- or no action.

## What the model should NOT be used for

The model should not be used to:

1. automatically publish content;
2. automatically rewrite titles or metadata;
3. automatically delete or prune pages;
4. automatically redirect pages;
5. automatically guarantee traffic recovery;
6. automatically claim that Google changed an algorithm;
7. replace human content or SEO review.

The model is a prioritization aid, not an autonomous production system.

## Limits of the evidence

The Week-6 grouped validation showed that validation design matters.

The observed Average Precision values were:

- Week-4 baseline: 0.5564
- Logistic Regression, random split: 0.6085
- Logistic Regression, client-grouped split: 0.5754

The grouped result was lower than the random-split result, which means the
random split gave a more optimistic estimate.

The grouped model still measured higher Average Precision than the Week-4
baseline, but the improvement was modest.

Therefore the appropriate interpretation is:

> The grouped validation measured a higher ranking score than the Week-4
> baseline, providing directional evidence that the model may improve review
> prioritization.

This is not evidence that a recommended content refresh will cause recovery.

## Scope of this queue

The current queue is generated from the validated ranking data available to
this notebook.

It should therefore be described as a **review queue derived from the
validated evaluation/ranking set**, not as a claim about every page in the
entire warehouse.

If a future version scores the full content inventory, that should be
described separately and validated again.

## Decision rule

The playbook uses the model score to determine review priority.

The score determines:

**priority**

The reason code determines:

**why the item deserves attention**

The human reviewer determines:

**what action, if any, should actually be taken**

This separation keeps the model, evidence, and business decision distinct.

## Public-safe claim

The safe claim for this project is:

> The model provides a ranked decision-support queue for prioritizing content
> review. It does not establish that a refresh will cause improved search
> performance.

All recommendations remain subject to human review.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 7 — SECTION 2
# INTENDED USE + LIMITS
# Lane 2: Refresh / Content Opportunity Scoring
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 75)
print("WEEK 7 — SECTION 2: INTENDED USE AND LIMITS")
print("=" * 75)


# ------------------------------------------------------------
# 1. Confirm Section 1 object exists
# ------------------------------------------------------------

if "queue_df" not in globals():
    raise RuntimeError(
        """
queue_df was not found.

Run SECTION 1 first.
Section 2 expects the ranked Week-5 output created there.
"""
    )

print("✅ Section 1 ranking object found.")

print("Rows available:", len(queue_df))


# ------------------------------------------------------------
# 2. Confirm the ranking score
# ------------------------------------------------------------

if "SCORE_COLUMN" not in globals():
    raise RuntimeError(
        """
SCORE_COLUMN was not found.

Run SECTION 1 first.
"""
    )

print("Ranking score:", SCORE_COLUMN)


# ------------------------------------------------------------
# 3. Confirm required identifiers
# ------------------------------------------------------------

required_columns = [
    "client_hash_id",
    "content_hash_id",
    "rank",
    SCORE_COLUMN
]

missing_columns = [
    col
    for col in required_columns
    if col not in queue_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\n" + "=" * 75)
print("QUEUE STRUCTURE CHECK")
print("=" * 75)

for col in required_columns:
    print("✅", col)


# ------------------------------------------------------------
# 4. Validate rank ordering
# ------------------------------------------------------------

score_is_descending = (
    queue_df[SCORE_COLUMN]
    .is_monotonic_decreasing
)

if not score_is_descending:
    raise ValueError(
        "Queue is not sorted from highest score to lowest score."
    )

print("\n✅ Queue is sorted by model score descending.")


# ------------------------------------------------------------
# 5. Validate rank sequence
# ------------------------------------------------------------

expected_ranks = np.arange(
    1,
    len(queue_df) + 1
)

actual_ranks = queue_df["rank"].to_numpy()

if not np.array_equal(
    expected_ranks,
    actual_ranks
):
    raise ValueError(
        "Rank column is not a continuous 1..N sequence."
    )

print("✅ Rank sequence is valid.")


# ------------------------------------------------------------
# 6. Create priority bands
# ------------------------------------------------------------
#
# IMPORTANT:
# These are NOT probability claims.
# They are operational priority bands.
#
# High:
#   top 20% of the available review queue
#
# Medium:
#   next 30%
#
# Monitor:
#   remaining 50%
#
# This avoids pretending that a score such as 0.70 means
# "70% chance of decline".

n = len(queue_df)

queue_df["priority_band"] = "MONITOR"

if n > 0:

    high_cutoff = max(
        1,
        int(np.ceil(n * 0.20))
    )

    medium_cutoff = max(
        high_cutoff,
        int(np.ceil(n * 0.50))
    )

    queue_df.loc[
        queue_df["rank"] <= high_cutoff,
        "priority_band"
    ] = "HIGH_PRIORITY"

    queue_df.loc[
        (queue_df["rank"] > high_cutoff)
        & (queue_df["rank"] <= medium_cutoff),
        "priority_band"
    ] = "MEDIUM_PRIORITY"


# ------------------------------------------------------------
# 7. Define intended-use actions
# ------------------------------------------------------------

action_map = {
    "HIGH_PRIORITY": "REFRESH_REVIEW",
    "MEDIUM_PRIORITY": "INVESTIGATE",
    "MONITOR": "MONITOR"
}

queue_df["recommended_action"] = (
    queue_df["priority_band"]
    .map(action_map)
)


# ------------------------------------------------------------
# 8. Create a simple explanation for the priority
# ------------------------------------------------------------

reason_map = {
    "HIGH_PRIORITY":
        "High model-priority score within the validated review queue.",

    "MEDIUM_PRIORITY":
        "Intermediate model-priority score; investigate before intervention.",

    "MONITOR":
        "Lower relative model-priority score; monitor rather than prioritize intervention."
}

queue_df["priority_reason"] = (
    queue_df["priority_band"]
    .map(reason_map)
)


# ------------------------------------------------------------
# 9. Explicitly check that this is NOT a causal action
# ------------------------------------------------------------

queue_df["human_review_required"] = True

queue_df["automation_allowed"] = False


# ------------------------------------------------------------
# 10. Display queue summary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PRIORITY BAND DISTRIBUTION")
print("=" * 75)

band_summary = (
    queue_df["priority_band"]
    .value_counts()
    .rename_axis("priority_band")
    .reset_index(name="n")
)

band_summary["percentage"] = (
    band_summary["n"]
    / len(queue_df)
    * 100
).round(2)

display(band_summary)


# ------------------------------------------------------------
# 11. Display the first 20 recommendations
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("TOP 20 HUMAN-REVIEW QUEUE")
print("=" * 75)

display(
    queue_df[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            SCORE_COLUMN,
            "priority_band",
            "recommended_action",
            "priority_reason",
            "human_review_required",
            "automation_allowed"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# 12. Safety checks
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("INTENDED-USE SAFETY CHECK")
print("=" * 75)

assert queue_df["human_review_required"].all()

assert not queue_df["automation_allowed"].any()

assert queue_df["recommended_action"].notna().all()

assert queue_df["priority_reason"].notna().all()

print("✅ Every recommendation requires human review.")
print("✅ No recommendation is marked for automatic execution.")
print("✅ Every item has an operational action.")
print("✅ Every item has an explanation for its priority.")


# ------------------------------------------------------------
# 13. Check for future variables
# ------------------------------------------------------------

future_columns = [
    "april_impressions",
    "april_clicks",
    "april_avg_position",
    "click_change_pct",
    "future_decline"
]

future_used = [
    col
    for col in future_columns
    if col in queue_df.columns
]

print("\n" + "=" * 75)
print("FUTURE INFORMATION CHECK")
print("=" * 75)

if future_used:
    print(
        "⚠️ Future/evaluation columns exist in queue_df:",
        future_used
    )
    print(
        "They are retained only as source/evaluation context "
        "and are NOT used to determine priority or action."
    )
else:
    print("✅ No future-window columns exist in the action queue.")


# ------------------------------------------------------------
# 14. Save Section-2 intermediate output
# ------------------------------------------------------------

output_dir = "/content/work/outputs"

os.makedirs(
    output_dir,
    exist_ok=True
)

section2_path = (
    output_dir
    + "/week7_intended_use_queue.csv"
)

queue_df.to_csv(
    section2_path,
    index=False
)

print("\n" + "=" * 75)
print("SECTION 2 COMPLETE")
print("=" * 75)

print("Rows in queue:", len(queue_df))

print(
    "Action distribution:"
)

print(
    queue_df["recommended_action"]
    .value_counts()
)

print(
    "\nSaved intermediate queue:"
)

print(section2_path)

print(
    "\nNext:"
    "\nSECTION 3 — HUMAN REVIEW + NO-GO LIST"
)

WEEK 7 — SECTION 2: INTENDED USE AND LIMITS
✅ Section 1 ranking object found.
Rows available: 106
Ranking score: logistic_score

QUEUE STRUCTURE CHECK
✅ client_hash_id
✅ content_hash_id
✅ rank
✅ logistic_score

✅ Queue is sorted by model score descending.
✅ Rank sequence is valid.

PRIORITY BAND DISTRIBUTION


,priority_band,n,percentage
0,MONITOR,53,50.00
1,MEDIUM_PRIORITY,31,29.25
2,HIGH_PRIORITY,22,20.75



TOP 20 HUMAN-REVIEW QUEUE


,rank,client_hash_id,content_hash_id,logistic_score,priority_band,recommended_action,priority_reason,human_review_required,automation_allowed
0,1,client_08a6a72ff48e62c0,content_1fbc173c6b0a0512,0.611929,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False
1,2,client_08a6a72ff48e62c0,content_67c22dec4f120ed8,0.608996,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False
2,3,client_3f0ce4d44fe94f3d,content_57195309e404a304,0.604387,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False
3,4,client_08a6a72ff48e62c0,content_78501978aafda9e4,0.600780,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False
4,5,client_a80fca3f171ed1de,content_95e0cd5a6f9d7391,0.600746,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False
5,6,client_0fa64a184f18a4a0,content_474c280cef704f95,0.599018,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False
6,7,client_0fa64a184f18a4a0,content_1a96c5be2c90593d,0.597749,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False
7,8,client_0fa64a184f18a4a0,content_50c98fc9745d64a3,0.595253,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False
8,9,client_08a6a72ff48e62c0,content_281440c260f75191,0.595245,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False
9,10,client_08a6a72ff48e62c0,content_5f2c5708acd38231,0.594797,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False



INTENDED-USE SAFETY CHECK
✅ Every recommendation requires human review.
✅ No recommendation is marked for automatic execution.
✅ Every item has an operational action.
✅ Every item has an explanation for its priority.

FUTURE INFORMATION CHECK
⚠️ Future/evaluation columns exist in queue_df: ['future_decline']
They are retained only as source/evaluation context and are NOT used to determine priority or action.

SECTION 2 COMPLETE
Rows in queue: 106
Action distribution:
recommended_action
MONITOR           53
INVESTIGATE       31
REFRESH_REVIEW    22
Name: count, dtype: int64

Saved intermediate queue:
/content/work/outputs/week7_intended_use_queue.csv

Next:
SECTION 3 — HUMAN REVIEW + NO-GO LIST


# 3. Human Review Rules and No-Go List

## Purpose

The Week-7 model output is a prioritization tool for human review.

The model ranks content according to its observed model score, but the score
does not determine the final business action.

A reviewer must examine the evidence before deciding whether a content item
should actually be refreshed, investigated, monitored, or left unchanged.

## Human-review workflow

The recommended workflow is:

1. Start with the highest-priority items in the ranked queue.
2. Inspect the model score and priority band.
3. Review the available search-performance evidence.
4. Read the reason code associated with the item.
5. Check whether there is a legitimate content or SEO reason for intervention.
6. Decide whether the item should be refreshed, investigated, monitored, or
   rejected.
7. Record the human decision and the reason for that decision.

## HIGH_PRIORITY — REFRESH_REVIEW

Items in the HIGH_PRIORITY band should receive the earliest human review.

The model is identifying these items as relatively high-priority within the
validated ranking set.

However, HIGH_PRIORITY does not mean "refresh automatically."

A reviewer should confirm that:

- the content is still relevant;
- the page has a legitimate opportunity for improvement;
- the observed search signals support investigation;
- there is no obvious reason why the page should remain unchanged;
- the proposed refresh is appropriate for the content.

The final decision remains with the human reviewer.

## MEDIUM_PRIORITY — INVESTIGATE

Medium-priority items should be investigated after the highest-priority
queue has been reviewed.

The purpose is to determine whether the model signal represents a meaningful
content opportunity or simply a ranking pattern that does not justify action.

A reviewer may decide to:

- investigate further;
- refresh later;
- monitor the page;
- or take no action.

## MONITOR

Lower-priority items should generally be monitored rather than immediately
changed.

Monitoring is appropriate when the evidence is not strong enough to justify
spending editorial or engineering resources.

A monitoring decision is not a statement that the content is healthy. It only
means that the item is currently lower priority in this queue.

## Human-review rules

The following rules apply to every recommendation:

### Rule 1 — Model score is not a final decision

The score determines priority, not the final action.

### Rule 2 — Human review is required

No content change should be automatically triggered by this notebook.

### Rule 3 — Check the evidence before acting

A reviewer should inspect the available search-performance evidence before
approving a refresh.

### Rule 4 — Consider business and content context

The model does not know every editorial, commercial, legal, or strategic
constraint.

Human context can override the model recommendation.

### Rule 5 — Document overrides

If a reviewer rejects a high-priority recommendation, the reason should be
recorded so that future model reviews can learn from recurring failure modes.

# No-Go List

The following actions should NOT be automated from this model:

1. Automatically rewriting page content.
2. Automatically changing titles or metadata.
3. Automatically deleting pages.
4. Automatically redirecting URLs.
5. Automatically publishing content.
6. Automatically changing canonical URLs.
7. Automatically declaring that a page has recovered.
8. Automatically claiming that a refresh caused traffic growth.
9. Automatically attributing performance changes to a search-engine algorithm
   update.
10. Automatically allocating unlimited editorial resources based only on the
    model score.

## Cases requiring extra caution

A human reviewer should be especially cautious when:

- the page is strategically important;
- the page is legally or commercially sensitive;
- the page has unusual traffic behavior;
- the available data appears incomplete;
- the model recommendation conflicts strongly with domain knowledge;
- the page has recently been changed;
- the content is intentionally low-volume;
- there is insufficient evidence to justify a refresh.

## Public-safe interpretation

The model should be described as a decision-support system.

The safe interpretation is:

> The model provides a ranked queue that helps prioritize human review of
> potential content opportunities.

It should not be described as:

> The model determines which pages must be refreshed.

The final content decision remains with a human reviewer.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 7 — SECTION 3
# HUMAN REVIEW RULES + NO-GO LIST
# Lane 2: Refresh / Content Opportunity Scoring
# ============================================================

import os
import pandas as pd

print("=" * 75)
print("WEEK 7 — SECTION 3: HUMAN REVIEW + NO-GO LIST")
print("=" * 75)


# ------------------------------------------------------------
# 1. Confirm Section 2 queue
# ------------------------------------------------------------

if "queue_df" not in globals():
    raise RuntimeError(
        """
queue_df was not found.

Run Sections 1 and 2 first.
"""
    )

print("✅ Section 2 queue found.")

print("Rows:", len(queue_df))


# ------------------------------------------------------------
# 2. Confirm required Section-2 columns
# ------------------------------------------------------------

required_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    SCORE_COLUMN,
    "priority_band",
    "recommended_action",
    "priority_reason",
    "human_review_required",
    "automation_allowed"
]

missing_columns = [
    col
    for col in required_columns
    if col not in queue_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Section-2 columns: {missing_columns}"
    )

print("\n" + "=" * 75)
print("SECTION 2 COLUMN CHECK")
print("=" * 75)

for col in required_columns:
    print("✅", col)


# ------------------------------------------------------------
# 3. Define explicit human-review rules
# ------------------------------------------------------------

review_rules = pd.DataFrame({
    "priority_band": [
        "HIGH_PRIORITY",
        "MEDIUM_PRIORITY",
        "MONITOR"
    ],

    "recommended_action": [
        "REFRESH_REVIEW",
        "INVESTIGATE",
        "MONITOR"
    ],

    "human_review_required": [
        True,
        True,
        True
    ],

    "automatic_action_allowed": [
        False,
        False,
        False
    ],

    "review_instruction": [
        "Review first and determine whether a refresh opportunity is supported.",
        "Investigate the evidence before deciding whether intervention is justified.",
        "Monitor unless additional evidence creates a stronger reason for intervention."
    ]
})

print("\n" + "=" * 75)
print("HUMAN REVIEW RULES")
print("=" * 75)

display(review_rules)


# ------------------------------------------------------------
# 4. Define explicit no-go actions
# ------------------------------------------------------------

no_go_actions = [
    "AUTOMATIC_CONTENT_REWRITE",
    "AUTOMATIC_TITLE_CHANGE",
    "AUTOMATIC_METADATA_CHANGE",
    "AUTOMATIC_PAGE_DELETION",
    "AUTOMATIC_REDIRECT",
    "AUTOMATIC_PUBLISH",
    "AUTOMATIC_CANONICAL_CHANGE",
    "AUTOMATIC_RECOVERY_CLAIM",
    "AUTOMATIC_CAUSAL_CLAIM",
    "AUTOMATIC_UNLIMITED_RESOURCE_ALLOCATION"
]

no_go_df = pd.DataFrame({
    "no_go_action": no_go_actions,
    "allowed": False,
    "reason": [
        "Requires human editorial review.",
        "Requires human SEO/content review.",
        "Requires human validation.",
        "Deletion is a high-impact irreversible action.",
        "Redirects can affect site behavior and require review.",
        "Publishing must not be automated by this model.",
        "Canonical decisions require technical/SEO review.",
        "The model does not prove recovery.",
        "The model does not establish causality.",
        "Resource allocation requires business context."
    ]
})

print("\n" + "=" * 75)
print("NO-GO LIST")
print("=" * 75)

display(no_go_df)


# ------------------------------------------------------------
# 5. Check every queue row requires human review
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("HUMAN REVIEW SAFETY CHECK")
print("=" * 75)

human_review_count = (
    queue_df["human_review_required"]
    .sum()
)

automation_count = (
    queue_df["automation_allowed"]
    .sum()
)

print(
    "Rows requiring human review:",
    human_review_count,
    "/",
    len(queue_df)
)

print(
    "Rows allowed for automatic execution:",
    automation_count
)

assert human_review_count == len(queue_df)

assert automation_count == 0

print("✅ Every queue item requires human review.")
print("✅ No queue item is allowed to execute automatically.")


# ------------------------------------------------------------
# 6. Check action mapping
# ------------------------------------------------------------

expected_action_map = {
    "HIGH_PRIORITY": "REFRESH_REVIEW",
    "MEDIUM_PRIORITY": "INVESTIGATE",
    "MONITOR": "MONITOR"
}

print("\n" + "=" * 75)
print("ACTION MAPPING CHECK")
print("=" * 75)

for band, expected_action in expected_action_map.items():

    observed_actions = (
        queue_df.loc[
            queue_df["priority_band"] == band,
            "recommended_action"
        ]
        .dropna()
        .unique()
        .tolist()
    )

    print(
        f"{band}:",
        observed_actions
    )

    if len(observed_actions) > 0:
        assert observed_actions == [expected_action]

print("✅ Priority-to-action mapping is consistent.")


# ------------------------------------------------------------
# 7. Create human-review queue
# ------------------------------------------------------------

review_queue = queue_df[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        SCORE_COLUMN,
        "priority_band",
        "recommended_action",
        "priority_reason",
        "human_review_required",
        "automation_allowed"
    ]
].copy()

review_queue["human_decision"] = "PENDING_REVIEW"

review_queue["human_override_reason"] = ""

review_queue["review_notes"] = ""


# ------------------------------------------------------------
# 8. Display first 20 review candidates
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FIRST 20 HUMAN REVIEW CANDIDATES")
print("=" * 75)

display(
    review_queue.head(20)
)


# ------------------------------------------------------------
# 9. Summary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SECTION 3 SUMMARY")
print("=" * 75)

print(
    "HIGH_PRIORITY:",
    (
        review_queue["priority_band"]
        == "HIGH_PRIORITY"
    ).sum()
)

print(
    "MEDIUM_PRIORITY:",
    (
        review_queue["priority_band"]
        == "MEDIUM_PRIORITY"
    ).sum()
)

print(
    "MONITOR:",
    (
        review_queue["priority_band"]
        == "MONITOR"
    ).sum()
)

print(
    "\nHuman review required:",
    review_queue["human_review_required"].all()
)

print(
    "Automatic execution allowed:",
    review_queue["automation_allowed"].any()
)


# ------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------

output_dir = "/content/work/outputs"

os.makedirs(
    output_dir,
    exist_ok=True
)

review_queue_path = (
    output_dir
    + "/week7_human_review_queue.csv"
)

no_go_path = (
    output_dir
    + "/week7_no_go_list.csv"
)

review_rules_path = (
    output_dir
    + "/week7_human_review_rules.csv"
)

review_queue.to_csv(
    review_queue_path,
    index=False
)

no_go_df.to_csv(
    no_go_path,
    index=False
)

review_rules.to_csv(
    review_rules_path,
    index=False
)

print("\n" + "=" * 75)
print("SECTION 3 COMPLETE")
print("=" * 75)

print("Saved:")
print(review_queue_path)
print(no_go_path)
print(review_rules_path)

print("\nNext:")
print("SECTION 4 — MONITORING / RETRAIN TRIGGERS")

WEEK 7 — SECTION 3: HUMAN REVIEW + NO-GO LIST
✅ Section 2 queue found.
Rows: 106

SECTION 2 COLUMN CHECK
✅ rank
✅ client_hash_id
✅ content_hash_id
✅ logistic_score
✅ priority_band
✅ recommended_action
✅ priority_reason
✅ human_review_required
✅ automation_allowed

HUMAN REVIEW RULES


,priority_band,recommended_action,human_review_required,automatic_action_allowed,review_instruction
0,HIGH_PRIORITY,REFRESH_REVIEW,True,False,Review first and determine whether a refresh o...
1,MEDIUM_PRIORITY,INVESTIGATE,True,False,Investigate the evidence before deciding wheth...
2,MONITOR,MONITOR,True,False,Monitor unless additional evidence creates a s...



NO-GO LIST


,no_go_action,allowed,reason
0,AUTOMATIC_CONTENT_REWRITE,False,Requires human editorial review.
1,AUTOMATIC_TITLE_CHANGE,False,Requires human SEO/content review.
2,AUTOMATIC_METADATA_CHANGE,False,Requires human validation.
3,AUTOMATIC_PAGE_DELETION,False,Deletion is a high-impact irreversible action.
4,AUTOMATIC_REDIRECT,False,Redirects can affect site behavior and require...
5,AUTOMATIC_PUBLISH,False,Publishing must not be automated by this model.
6,AUTOMATIC_CANONICAL_CHANGE,False,Canonical decisions require technical/SEO review.
7,AUTOMATIC_RECOVERY_CLAIM,False,The model does not prove recovery.
8,AUTOMATIC_CAUSAL_CLAIM,False,The model does not establish causality.
9,AUTOMATIC_UNLIMITED_RESOURCE_ALLOCATION,False,Resource allocation requires business context.



HUMAN REVIEW SAFETY CHECK
Rows requiring human review: 106 / 106
Rows allowed for automatic execution: 0
✅ Every queue item requires human review.
✅ No queue item is allowed to execute automatically.

ACTION MAPPING CHECK
HIGH_PRIORITY: ['REFRESH_REVIEW']
MEDIUM_PRIORITY: ['INVESTIGATE']
MONITOR: ['MONITOR']
✅ Priority-to-action mapping is consistent.

FIRST 20 HUMAN REVIEW CANDIDATES


,rank,client_hash_id,content_hash_id,logistic_score,priority_band,recommended_action,priority_reason,human_review_required,automation_allowed,human_decision,human_override_reason,review_notes
0,1,client_08a6a72ff48e62c0,content_1fbc173c6b0a0512,0.611929,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False,PENDING_REVIEW,,
1,2,client_08a6a72ff48e62c0,content_67c22dec4f120ed8,0.608996,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False,PENDING_REVIEW,,
2,3,client_3f0ce4d44fe94f3d,content_57195309e404a304,0.604387,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False,PENDING_REVIEW,,
3,4,client_08a6a72ff48e62c0,content_78501978aafda9e4,0.600780,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False,PENDING_REVIEW,,
4,5,client_a80fca3f171ed1de,content_95e0cd5a6f9d7391,0.600746,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False,PENDING_REVIEW,,
5,6,client_0fa64a184f18a4a0,content_474c280cef704f95,0.599018,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False,PENDING_REVIEW,,
6,7,client_0fa64a184f18a4a0,content_1a96c5be2c90593d,0.597749,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False,PENDING_REVIEW,,
7,8,client_0fa64a184f18a4a0,content_50c98fc9745d64a3,0.595253,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False,PENDING_REVIEW,,
8,9,client_08a6a72ff48e62c0,content_281440c260f75191,0.595245,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False,PENDING_REVIEW,,
9,10,client_08a6a72ff48e62c0,content_5f2c5708acd38231,0.594797,HIGH_PRIORITY,REFRESH_REVIEW,High model-priority score within the validated...,True,False,PENDING_REVIEW,,



SECTION 3 SUMMARY
HIGH_PRIORITY: 22
MEDIUM_PRIORITY: 31
MONITOR: 53

Human review required: True
Automatic execution allowed: False

SECTION 3 COMPLETE
Saved:
/content/work/outputs/week7_human_review_queue.csv
/content/work/outputs/week7_no_go_list.csv
/content/work/outputs/week7_human_review_rules.csv

Next:
SECTION 4 — MONITORING / RETRAIN TRIGGERS


# 4. Monitoring and Retrain Triggers

## Purpose

The Week-7 action playbook is a decision-support system, so the model should
not be considered permanently valid just because it performed well during
validation.

The purpose of monitoring is to detect meaningful changes in:

- the input data;
- model-score distributions;
- priority-band distributions;
- observed outcome performance;
- and the quality of human-reviewed recommendations.

Monitoring should trigger investigation first. It should not automatically
trigger retraining.

## What should be monitored?

### 1. Data quality

The March feature inputs should continue to contain the expected fields:

- `march_impressions`
- `march_clicks`
- `march_avg_position`

We should check for:

- missing values;
- unexpected data types;
- infinite values;
- unexpected negative values;
- major changes in the number of records.

A data-quality problem should be investigated before judging model
performance.

### 2. Model-score distribution

The distribution of `logistic_score` should be monitored over time.

A substantial change in the score distribution can indicate that the
population or input signals have changed.

This is a monitoring signal, not proof that the model has failed.

### 3. Priority-band distribution

The proportion of items in:

- `HIGH_PRIORITY`
- `MEDIUM_PRIORITY`
- `MONITOR`

should be monitored.

A large shift may indicate a change in the incoming content population or
the model's behavior.

### 4. Observed model performance

When the future outcome becomes available, the model should be evaluated
using the same type of ranking/performance metric used during validation.

The Week-6 grouped validation result provides the current reference point:

- Week-4 baseline Average Precision: approximately 0.5564
- Week-5 Logistic Regression grouped Average Precision: approximately 0.5754

These numbers are reference evidence, not permanent guarantees.

## Retrain triggers

Retraining should be considered only after evidence shows that the model is
no longer providing useful decision-support.

Examples of triggers include:

1. Sustained deterioration in observed ranking performance.
2. Meaningful and persistent change in the input population.
3. Persistent changes in model-score or priority-band distributions.
4. Repeated human-review overrides indicating systematic model errors.
5. A change in the business problem, target definition, or available signals.

A single unusual observation should not automatically trigger retraining.

## Human review remains part of monitoring

Human-review outcomes should be recorded.

Examples include:

- recommendation accepted;
- recommendation rejected;
- recommendation changed to another action;
- recommendation considered insufficiently supported.

Repeated patterns in these decisions can reveal weaknesses that aggregate
model metrics may not show.

## What should NOT happen automatically

The following should not happen automatically:

- automatic model retraining;
- automatic content changes;
- automatic publishing;
- automatic threshold changes;
- automatic claims that model performance has degraded;
- automatic claims that a content refresh caused recovery.

A monitoring trigger should create an investigation or review step.

## Practical monitoring cycle

The intended operating cycle is:

Data quality check
→ score distribution check
→ priority distribution check
→ observed outcome evaluation
→ human-review feedback
→ investigation if needed
→ retraining decision only when evidence supports it.

## Public-safe interpretation

The playbook should describe monitoring as a way to detect possible changes
in model usefulness.

It should use language such as:

- "monitor"
- "observed"
- "measured"
- "investigate"
- "consider retraining"

It should avoid language such as:

- "guarantees"
- "will recover traffic"
- "automatically detects failure"
- "proves the refresh caused recovery"

The model remains decision-support rather than an autonomous production
system.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 7 — SECTION 4
# MONITORING / RETRAIN TRIGGERS
# Lane 2: Refresh / Content Opportunity Scoring
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 75)
print("WEEK 7 — SECTION 4: MONITORING / RETRAIN TRIGGERS")
print("=" * 75)


# ------------------------------------------------------------
# 1. Confirm Section 3 objects
# ------------------------------------------------------------

if "queue_df" not in globals():
    raise RuntimeError(
        """
queue_df was not found.

Run Sections 1, 2 and 3 before running Section 4.
"""
    )

print("✅ Section 3 queue found.")
print("Rows:", len(queue_df))


# ------------------------------------------------------------
# 2. Confirm score column
# ------------------------------------------------------------

if "SCORE_COLUMN" not in globals():
    SCORE_COLUMN = "logistic_score"

if SCORE_COLUMN not in queue_df.columns:
    raise ValueError(
        f"Ranking score column '{SCORE_COLUMN}' was not found."
    )

print("Ranking score:", SCORE_COLUMN)


# ------------------------------------------------------------
# 3. Required monitoring columns
# ------------------------------------------------------------

required_columns = [
    "client_hash_id",
    "content_hash_id",
    SCORE_COLUMN,
    "priority_band",
    "recommended_action"
]

missing_columns = [
    col
    for col in required_columns
    if col not in queue_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing monitoring columns: {missing_columns}"
    )

print("\n" + "=" * 75)
print("MONITORING INPUT CHECK")
print("=" * 75)

for col in required_columns:
    print("✅", col)


# ------------------------------------------------------------
# 4. Score quality checks
# ------------------------------------------------------------

scores = pd.to_numeric(
    queue_df[SCORE_COLUMN],
    errors="coerce"
)

print("\n" + "=" * 75)
print("MODEL SCORE QUALITY")
print("=" * 75)

print("Total rows:", len(scores))
print("Missing scores:", scores.isna().sum())
print("Infinite scores:", np.isinf(scores).sum())

assert scores.notna().all()
assert np.isfinite(scores).all()

print("Minimum score:", round(scores.min(), 6))
print("Maximum score:", round(scores.max(), 6))
print("Mean score:", round(scores.mean(), 6))
print("Median score:", round(scores.median(), 6))

print("✅ No missing or infinite model scores.")


# ------------------------------------------------------------
# 5. Score distribution reference
# ------------------------------------------------------------

score_summary = pd.DataFrame({
    "metric": [
        "count",
        "mean",
        "std",
        "min",
        "25%",
        "50%",
        "75%",
        "max"
    ],
    "value": [
        scores.count(),
        scores.mean(),
        scores.std(),
        scores.min(),
        scores.quantile(0.25),
        scores.quantile(0.50),
        scores.quantile(0.75),
        scores.max()
    ]
})

print("\n" + "=" * 75)
print("CURRENT SCORE DISTRIBUTION REFERENCE")
print("=" * 75)

display(score_summary)


# ------------------------------------------------------------
# 6. Priority-band monitoring
# ------------------------------------------------------------

priority_counts = (
    queue_df["priority_band"]
    .value_counts()
    .reindex(
        [
            "HIGH_PRIORITY",
            "MEDIUM_PRIORITY",
            "MONITOR"
        ],
        fill_value=0
    )
)

priority_monitor = pd.DataFrame({
    "priority_band": priority_counts.index,
    "n": priority_counts.values,
    "percentage": (
        priority_counts.values
        / len(queue_df)
        * 100
    )
})

print("\n" + "=" * 75)
print("PRIORITY-BAND DISTRIBUTION")
print("=" * 75)

display(priority_monitor)


# ------------------------------------------------------------
# 7. Action distribution
# ------------------------------------------------------------

action_counts = (
    queue_df["recommended_action"]
    .value_counts()
)

action_monitor = pd.DataFrame({
    "recommended_action": action_counts.index,
    "n": action_counts.values,
    "percentage": (
        action_counts.values
        / len(queue_df)
        * 100
    )
})

print("\n" + "=" * 75)
print("ACTION DISTRIBUTION")
print("=" * 75)

display(action_monitor)


# ------------------------------------------------------------
# 8. Data-quality checks for identifiers
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("IDENTIFIER / DATA QUALITY CHECK")
print("=" * 75)

print(
    "Missing client IDs:",
    queue_df["client_hash_id"].isna().sum()
)

print(
    "Missing content IDs:",
    queue_df["content_hash_id"].isna().sum()
)

print(
    "Duplicate client-content pairs:",
    queue_df.duplicated(
        subset=[
            "client_hash_id",
            "content_hash_id"
        ]
    ).sum()
)

assert queue_df["client_hash_id"].notna().all()
assert queue_df["content_hash_id"].notna().all()

print("✅ Required identifiers are present.")


# ------------------------------------------------------------
# 9. Define monitoring triggers
# ------------------------------------------------------------

monitoring_triggers = pd.DataFrame({
    "trigger": [
        "Data quality degradation",
        "Score distribution shift",
        "Priority-band distribution shift",
        "Sustained performance deterioration",
        "Repeated human-review overrides",
        "Target or business-definition change"
    ],

    "what_to_monitor": [
        "Missing values, invalid values, record counts and schema.",
        "Mean, median and percentile distribution of model scores.",
        "Share of HIGH_PRIORITY, MEDIUM_PRIORITY and MONITOR items.",
        "Observed ranking performance after the future outcome is available.",
        "Repeated rejection or override patterns in human review.",
        "Changes to the target, business objective or available signals."
    ],

    "response": [
        "Investigate data pipeline before evaluating model performance.",
        "Investigate population or feature changes.",
        "Investigate whether the incoming content population has changed.",
        "Investigate model usefulness and validation results.",
        "Investigate systematic failure modes.",
        "Revisit the modeling and validation design."
    ],

    "automatic_retrain": [
        False,
        False,
        False,
        False,
        False,
        False
    ]
})

print("\n" + "=" * 75)
print("MONITORING / RETRAIN TRIGGERS")
print("=" * 75)

display(monitoring_triggers)


# ------------------------------------------------------------
# 10. Explicit no-automatic-retrain check
# ------------------------------------------------------------

assert (
    monitoring_triggers["automatic_retrain"]
    .eq(False)
    .all()
)

print("\n" + "=" * 75)
print("AUTOMATION SAFETY CHECK")
print("=" * 75)

print(
    "Automatic retraining triggers:",
    monitoring_triggers["automatic_retrain"].sum()
)

print("✅ No monitoring condition automatically retrains the model.")
print("✅ Monitoring triggers investigation rather than automatic action.")


# ------------------------------------------------------------
# 11. Reference evidence from Week 6
# ------------------------------------------------------------

reference_metrics = pd.DataFrame({
    "system": [
        "Week-4 Baseline",
        "Week-5 Logistic Regression — Grouped"
    ],

    "average_precision": [
        0.5564,
        0.5754
    ],

    "role": [
        "Historical baseline reference",
        "Current grouped-validation reference"
    ]
})

print("\n" + "=" * 75)
print("CURRENT VALIDATION REFERENCE")
print("=" * 75)

display(reference_metrics)


# ------------------------------------------------------------
# 12. Create a monitoring receipt
# ------------------------------------------------------------

monitoring_receipt = pd.DataFrame({
    "metric": [
        "queue_rows",
        "score_mean",
        "score_median",
        "score_min",
        "score_max",
        "high_priority_pct",
        "medium_priority_pct",
        "monitor_pct",
        "week4_baseline_ap",
        "week5_grouped_ap"
    ],

    "value": [
        len(queue_df),
        scores.mean(),
        scores.median(),
        scores.min(),
        scores.max(),

        (
            priority_counts["HIGH_PRIORITY"]
            / len(queue_df)
            * 100
        ),

        (
            priority_counts["MEDIUM_PRIORITY"]
            / len(queue_df)
            * 100
        ),

        (
            priority_counts["MONITOR"]
            / len(queue_df)
            * 100
        ),

        0.5564,
        0.5754
    ]
})

print("\n" + "=" * 75)
print("MONITORING RECEIPT")
print("=" * 75)

display(monitoring_receipt)


# ------------------------------------------------------------
# 13. Save outputs
# ------------------------------------------------------------

output_dir = "/content/work/outputs"

os.makedirs(
    output_dir,
    exist_ok=True
)

monitoring_path = (
    output_dir
    + "/week7_monitoring_reference.csv"
)

triggers_path = (
    output_dir
    + "/week7_monitoring_triggers.csv"
)

priority_path = (
    output_dir
    + "/week7_priority_distribution.csv"
)

monitoring_receipt.to_csv(
    monitoring_path,
    index=False
)

monitoring_triggers.to_csv(
    triggers_path,
    index=False
)

priority_monitor.to_csv(
    priority_path,
    index=False
)

print("\n" + "=" * 75)
print("SECTION 4 COMPLETE")
print("=" * 75)

print("Saved:")
print(monitoring_path)
print(triggers_path)
print(priority_path)

print("\nKey principle:")
print(
    "Monitor → investigate → validate evidence → "
    "consider retraining."
)

print("\nNext:")
print("SECTION 5 — EXPORTS FOR THE PAPER")

WEEK 7 — SECTION 4: MONITORING / RETRAIN TRIGGERS
✅ Section 3 queue found.
Rows: 106
Ranking score: logistic_score

MONITORING INPUT CHECK
✅ client_hash_id
✅ content_hash_id
✅ logistic_score
✅ priority_band
✅ recommended_action

MODEL SCORE QUALITY
Total rows: 106
Missing scores: 0
Infinite scores: 0
Minimum score: 0.082823
Maximum score: 0.611929
Mean score: 0.5524
Median score: 0.570166
✅ No missing or infinite model scores.

CURRENT SCORE DISTRIBUTION REFERENCE


,metric,value
0,count,106.000000
1,mean,0.552400
2,std,0.061676
3,min,0.082823
4,25%,0.533370
5,50%,0.570166
6,75%,0.586224
7,max,0.611929



PRIORITY-BAND DISTRIBUTION


,priority_band,n,percentage
0,HIGH_PRIORITY,22,20.754717
1,MEDIUM_PRIORITY,31,29.245283
2,MONITOR,53,50.000000



ACTION DISTRIBUTION


...


AUTOMATION SAFETY CHECK
Automatic retraining triggers: 0
✅ No monitoring condition automatically retrains the model.
✅ Monitoring triggers investigation rather than automatic action.

CURRENT VALIDATION REFERENCE


,system,average_precision,role
0,Week-4 Baseline,0.5564,Historical baseline reference
1,Week-5 Logistic Regression — Grouped,0.5754,Current grouped-validation reference



MONITORING RECEIPT


,metric,value
0,queue_rows,106.000000
1,score_mean,0.552400
2,score_median,0.570166
3,score_min,0.082823
4,score_max,0.611929
5,high_priority_pct,20.754717
6,medium_priority_pct,29.245283
7,monitor_pct,50.000000
8,week4_baseline_ap,0.556400
9,week5_grouped_ap,0.575400



SECTION 4 COMPLETE
Saved:
/content/work/outputs/week7_monitoring_reference.csv
/content/work/outputs/week7_monitoring_triggers.csv
/content/work/outputs/week7_priority_distribution.csv

Key principle:
Monitor → investigate → validate evidence → consider retraining.

Next:
SECTION 5 — EXPORTS FOR THE PAPER


# 5. Exports for the Paper

## Purpose

This section creates the artifacts that will support the recommendations
section of the research paper.

The exports are generated from the validated Week-7 action playbook rather
than being manually copied from notebook output.

The main paper-facing artifacts are:

1. The ranked action queue.
2. The human-review rules.
3. The no-go list.
4. The monitoring and retraining triggers.
5. The monitoring reference metrics.
6. A compact summary of the model evidence and intended use.

## Ranked action queue

The ranked queue contains the model-prioritized content items together with:

- rank;
- client identifier;
- content identifier;
- model score;
- priority band;
- recommended action;
- reason for priority;
- human-review requirement;
- automation status.

The queue is intended to support human prioritization.

It should not be interpreted as an automatically executable content plan.

## Paper evidence

The Week-6 validation provides the main evidence supporting the action
playbook:

- Week-4 baseline Average Precision: 0.5564
- Week-5 Logistic Regression grouped Average Precision: 0.5754
- Random-split Average Precision: 0.6085

The grouped result is preferred when describing expected generalization because
the grouped split prevents clients from appearing in both training and test
sets.

The grouped result was higher than the Week-4 baseline, but the improvement
should be described as measured or observed evidence rather than proof of
business impact.

## Human review policy

The exported review policy makes clear that:

- HIGH_PRIORITY means review first;
- MEDIUM_PRIORITY means investigate;
- MONITOR means monitor rather than prioritize intervention;
- all recommendations require human review;
- automatic execution is disabled.

## No-go policy

The exported no-go list documents actions that should not be automated from
this model, including automatic content changes, publishing, deletion,
redirects, recovery claims, and causal claims.

## Monitoring

The exported monitoring artifacts provide a reference snapshot for future
comparisons.

Future monitoring should compare new observations against these references
rather than treating the current snapshot as proof of drift.

## Public-safe interpretation

The final paper should describe the system as a decision-support playbook.

A safe summary is:

> The validated model provides a ranked queue for prioritizing human review
> of potential content opportunities. The playbook defines review rules,
> limits automation, and specifies monitoring signals for future evaluation.

The exports in this section are evidence and implementation artifacts for the
paper. They do not establish that a content refresh causes improved search
performance.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 7 — SECTION 5
# EXPORTS FOR THE PAPER
# Lane 2: Refresh / Content Opportunity Scoring
# ============================================================

import os
import json
import numpy as np
import pandas as pd

print("=" * 75)
print("WEEK 7 — SECTION 5: EXPORTS FOR THE PAPER")
print("=" * 75)


# ------------------------------------------------------------
# 1. Confirm required objects
# ------------------------------------------------------------

required_objects = [
    "queue_df",
    "review_queue",
    "no_go_df",
    "review_rules",
    "monitoring_triggers",
    "monitoring_receipt",
    "priority_monitor"
]

print("\n" + "=" * 75)
print("OBJECT CHECK")
print("=" * 75)

missing_objects = []

for obj in required_objects:

    if obj in globals():
        print("✅", obj)
    else:
        print("❌", obj)
        missing_objects.append(obj)

if missing_objects:

    raise RuntimeError(
        "Missing required Section-2/3/4 objects: "
        + str(missing_objects)
    )


# ------------------------------------------------------------
# 2. Create output directories
# ------------------------------------------------------------

output_dir = "/content/work/outputs"
figures_dir = "/content/work/figures"

os.makedirs(
    output_dir,
    exist_ok=True
)

os.makedirs(
    figures_dir,
    exist_ok=True
)

print("\n✅ Output directories ready.")


# ------------------------------------------------------------
# 3. Build final paper-facing ranked queue
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BUILDING FINAL RANKED ACTION QUEUE")
print("=" * 75)

paper_queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    SCORE_COLUMN,
    "priority_band",
    "recommended_action",
    "priority_reason",
    "human_review_required",
    "automation_allowed"
]

missing_queue_columns = [
    col
    for col in paper_queue_columns
    if col not in queue_df.columns
]

if missing_queue_columns:

    raise ValueError(
        "Missing queue columns: "
        + str(missing_queue_columns)
    )

paper_queue = (
    queue_df[
        paper_queue_columns
    ]
    .copy()
    .sort_values(
        "rank"
    )
    .reset_index(
        drop=True
    )
)

print(
    "Rows in final ranked queue:",
    len(paper_queue)
)

print(
    "Ranking score:",
    SCORE_COLUMN
)


# ------------------------------------------------------------
# 4. Safety check — queue ordering
# ------------------------------------------------------------

assert (
    paper_queue["rank"]
    .is_monotonic_increasing
)

assert (
    paper_queue[SCORE_COLUMN]
    .is_monotonic_decreasing
)

assert (
    paper_queue["human_review_required"]
    .all()
)

assert not (
    paper_queue["automation_allowed"]
    .any()
)

print("✅ Queue is correctly ranked.")
print("✅ Human review is required for every row.")
print("✅ Automatic execution is disabled.")


# ------------------------------------------------------------
# 5. Export ranked queue
# ------------------------------------------------------------

ranked_queue_path = (
    output_dir
    + "/week7_ranked_action_queue.csv"
)

paper_queue.to_csv(
    ranked_queue_path,
    index=False
)

print(
    "\n✅ Ranked action queue exported:"
)

print(ranked_queue_path)


# ------------------------------------------------------------
# 6. Export human-review rules
# ------------------------------------------------------------

review_rules_path = (
    output_dir
    + "/week7_human_review_rules.csv"
)

review_rules.to_csv(
    review_rules_path,
    index=False
)

print(
    "✅ Human-review rules exported:"
)

print(review_rules_path)


# ------------------------------------------------------------
# 7. Export no-go list
# ------------------------------------------------------------

no_go_path = (
    output_dir
    + "/week7_no_go_list.csv"
)

no_go_df.to_csv(
    no_go_path,
    index=False
)

print(
    "✅ No-go list exported:"
)

print(no_go_path)


# ------------------------------------------------------------
# 8. Export monitoring triggers
# ------------------------------------------------------------

monitoring_triggers_path = (
    output_dir
    + "/week7_monitoring_triggers.csv"
)

monitoring_triggers.to_csv(
    monitoring_triggers_path,
    index=False
)

print(
    "✅ Monitoring triggers exported:"
)

print(monitoring_triggers_path)


# ------------------------------------------------------------
# 9. Export monitoring reference
# ------------------------------------------------------------

monitoring_reference_path = (
    output_dir
    + "/week7_monitoring_reference.csv"
)

monitoring_receipt.to_csv(
    monitoring_reference_path,
    index=False
)

print(
    "✅ Monitoring reference exported:"
)

print(monitoring_reference_path)


# ------------------------------------------------------------
# 10. Export priority distribution
# ------------------------------------------------------------

priority_distribution_path = (
    output_dir
    + "/week7_priority_distribution.csv"
)

priority_monitor.to_csv(
    priority_distribution_path,
    index=False
)

print(
    "✅ Priority distribution exported:"
)

print(priority_distribution_path)


# ------------------------------------------------------------
# 11. Create paper evidence summary
# ------------------------------------------------------------

paper_evidence = pd.DataFrame({

    "evidence_item": [

        "decision_window",
        "outcome_window",
        "lane",
        "ranking_model",
        "queue_rows",
        "high_priority_rows",
        "medium_priority_rows",
        "monitor_rows",
        "week4_baseline_average_precision",
        "week5_random_split_average_precision",
        "week5_grouped_average_precision",
        "grouped_minus_baseline",
        "grouped_minus_random",
        "human_review_required",
        "automatic_execution_allowed"
    ],

    "value": [

        "March 2026",
        "April 2026",
        "Lane 2 — Refresh / Content Opportunity Scoring",
        SCORE_COLUMN,
        len(paper_queue),

        int(
            (
                paper_queue["priority_band"]
                == "HIGH_PRIORITY"
            ).sum()
        ),

        int(
            (
                paper_queue["priority_band"]
                == "MEDIUM_PRIORITY"
            ).sum()
        ),

        int(
            (
                paper_queue["priority_band"]
                == "MONITOR"
            ).sum()
        ),

        0.5564,
        0.6085,
        0.5754,

        0.5754 - 0.5564,
        0.5754 - 0.6085,

        True,
        False
    ]
})

print("\n" + "=" * 75)
print("PAPER EVIDENCE SUMMARY")
print("=" * 75)

display(
    paper_evidence
)


# ------------------------------------------------------------
# 12. Export paper evidence summary
# ------------------------------------------------------------

paper_evidence_path = (
    output_dir
    + "/week7_paper_evidence_summary.csv"
)

paper_evidence.to_csv(
    paper_evidence_path,
    index=False
)

print(
    "\n✅ Paper evidence summary exported:"
)

print(paper_evidence_path)


# ------------------------------------------------------------
# 13. Create compact JSON receipt
# ------------------------------------------------------------

json_receipt = {

    "lane":
        "Lane 2 — Refresh / Content Opportunity Scoring",

    "decision_window":
        "March 2026",

    "observed_outcome_window":
        "April 2026",

    "ranking_model":
        SCORE_COLUMN,

    "queue_rows":
        int(len(paper_queue)),

    "priority_distribution": {

        "HIGH_PRIORITY":
            int(
                (
                    paper_queue["priority_band"]
                    == "HIGH_PRIORITY"
                ).sum()
            ),

        "MEDIUM_PRIORITY":
            int(
                (
                    paper_queue["priority_band"]
                    == "MEDIUM_PRIORITY"
                ).sum()
            ),

        "MONITOR":
            int(
                (
                    paper_queue["priority_band"]
                    == "MONITOR"
                ).sum()
            )
    },

    "validation": {

        "week4_baseline_average_precision":
            0.5564,

        "week5_random_split_average_precision":
            0.6085,

        "week5_grouped_average_precision":
            0.5754
    },

    "decision_support": {

        "human_review_required":
            True,

        "automatic_execution_allowed":
            False
    },

    "interpretation":
        "Observed ranking evidence for decision-support; "
        "not causal evidence that content refresh causes recovery."
}

json_path = (
    output_dir
    + "/week7_paper_evidence.json"
)

with open(
    json_path,
    "w"
) as f:

    json.dump(
        json_receipt,
        f,
        indent=2
    )

print(
    "✅ Paper evidence JSON exported:"
)

print(json_path)


# ------------------------------------------------------------
# 14. Create a paper-ready text summary
# ------------------------------------------------------------

paper_summary = f"""
WEEK 7 ACTION PLAYBOOK — PAPER SUMMARY

Lane:
Lane 2 — Refresh / Content Opportunity Scoring

Decision window:
March 2026

Observed outcome window:
April 2026

Ranking model:
{SCORE_COLUMN}

Validated ranking queue:
{len(paper_queue)} items

Priority distribution:
- HIGH_PRIORITY: {
    int(
        (
            paper_queue["priority_band"]
            == "HIGH_PRIORITY"
        ).sum()
    )
}
- MEDIUM_PRIORITY: {
    int(
        (
            paper_queue["priority_band"]
            == "MEDIUM_PRIORITY"
        ).sum()
    )
}
- MONITOR: {
    int(
        (
            paper_queue["priority_band"]
            == "MONITOR"
        ).sum()
    )
}

Validation evidence:
- Week-4 baseline Average Precision: 0.5564
- Week-5 random-split Average Precision: 0.6085
- Week-5 client-grouped Average Precision: 0.5754

Grouped model minus baseline:
+0.0190

Grouped model minus random split:
-0.0331

Intended use:
The model provides a ranked decision-support queue for prioritizing human
review of potential content opportunities.

Human review:
Required for every recommendation.

Automatic execution:
Disabled.

Important limitation:
The model provides observed ranking evidence. It does not establish that
refreshing a page will cause improved search performance.

Monitoring:
Monitor data quality, score distributions, priority distributions, observed
future performance, and repeated human-review overrides.

Retraining:
Retraining should be considered after evidence of sustained degradation,
meaningful population change, repeated systematic errors, or a change in the
target/business problem. Monitoring does not automatically retrain the model.
"""

summary_path = (
    output_dir
    + "/week7_paper_summary.txt"
)

with open(
    summary_path,
    "w"
) as f:

    f.write(
        paper_summary.strip()
    )

print(
    "✅ Paper summary exported:"
)

print(summary_path)


# ------------------------------------------------------------
# 15. Final export inventory
# ------------------------------------------------------------

exported_files = [
    ranked_queue_path,
    review_rules_path,
    no_go_path,
    monitoring_triggers_path,
    monitoring_reference_path,
    priority_distribution_path,
    paper_evidence_path,
    json_path,
    summary_path
]

print("\n" + "=" * 75)
print("SECTION 5 EXPORT INVENTORY")
print("=" * 75)

for path in exported_files:

    exists = os.path.exists(path)

    print(
        "✅" if exists else "❌",
        path
    )

    assert exists


# ------------------------------------------------------------
# 16. Final consistency checks
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL EXPORT CONSISTENCY CHECK")
print("=" * 75)

assert len(paper_queue) == len(queue_df)

assert (
    paper_queue["rank"]
    .iloc[0]
    == 1
)

assert (
    paper_queue["rank"]
    .iloc[-1]
    == len(paper_queue)
)

assert (
    paper_queue["human_review_required"]
    .all()
)

assert not (
    paper_queue["automation_allowed"]
    .any()
)

print("✅ Ranked queue row count matches source queue.")
print("✅ Rank starts at 1.")
print("✅ Rank ends at queue size.")
print("✅ Human review is required.")
print("✅ Automatic execution is disabled.")
print("✅ Paper evidence exports are complete.")


# ------------------------------------------------------------
# 17. Final status
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SECTION 5 COMPLETE")
print("=" * 75)

print(
    """
The Week-7 paper-facing artifacts have been generated.

The ranked queue, human-review policy, no-go list, monitoring references,
monitoring triggers, evidence summary, JSON receipt and paper summary
are available under:

/content/work/outputs/

The queue remains a non-production decision-support artifact.
"""
)

print("Next:")
print("SECTION 6 — SELF-CHECK")

WEEK 7 — SECTION 5: EXPORTS FOR THE PAPER

OBJECT CHECK
✅ queue_df
✅ review_queue
✅ no_go_df
✅ review_rules
✅ monitoring_triggers
✅ monitoring_receipt
✅ priority_monitor

✅ Output directories ready.

BUILDING FINAL RANKED ACTION QUEUE
Rows in final ranked queue: 106
Ranking score: logistic_score
✅ Queue is correctly ranked.
✅ Human review is required for every row.
✅ Automatic execution is disabled.

✅ Ranked action queue exported:
/content/work/outputs/week7_ranked_action_queue.csv
✅ Human-review rules exported:
/content/work/outputs/week7_human_review_rules.csv
✅ No-go list exported:
/content/work/outputs/week7_no_go_list.csv
✅ Monitoring triggers exported:
/content/work/outputs/week7_monitoring_triggers.csv
✅ Monitoring reference exported:
/content/work/outputs/week7_monitoring_reference.csv
✅ Priority distribution exported:
/content/work/outputs/week7_priority_distribution.csv

PAPER EVIDENCE SUMMARY


,evidence_item,value
0,decision_window,March 2026
1,outcome_window,April 2026
2,lane,Lane 2 — Refresh / Content Opportunity Scoring
3,ranking_model,logistic_score
4,queue_rows,106
5,high_priority_rows,22
6,medium_priority_rows,31
7,monitor_rows,53
8,week4_baseline_average_precision,0.5564
9,week5_random_split_average_precision,0.6085



✅ Paper evidence summary exported:
/content/work/outputs/week7_paper_evidence_summary.csv
✅ Paper evidence JSON exported:
/content/work/outputs/week7_paper_evidence.json
✅ Paper summary exported:
/content/work/outputs/week7_paper_summary.txt

SECTION 5 EXPORT INVENTORY
✅ /content/work/outputs/week7_ranked_action_queue.csv
✅ /content/work/outputs/week7_human_review_rules.csv
✅ /content/work/outputs/week7_no_go_list.csv
✅ /content/work/outputs/week7_monitoring_triggers.csv
✅ /content/work/outputs/week7_monitoring_reference.csv
✅ /content/work/outputs/week7_priority_distribution.csv
✅ /content/work/outputs/week7_paper_evidence_summary.csv
✅ /content/work/outputs/week7_paper_evidence.json
✅ /content/work/outputs/week7_paper_summary.txt

FINAL EXPORT CONSISTENCY CHECK
✅ Ranked queue row count matches source queue.
✅ Rank starts at 1.
✅ Rank ends at queue size.
✅ Human review is required.
✅ Automatic execution is disabled.
✅ Paper evidence exports are complete.

SECTION 5 COMPLETE

The Week

# 6. Self-Check

This final section verifies that the Week-7 action playbook is internally
consistent and ready for review.

The self-check confirms:

1. The Week-5 ranking evidence was used to create the Week-7 queue.
2. The queue is correctly ranked by the selected model score.
3. Every recommendation has a defined action and reason.
4. Human review is required for every recommendation.
5. Automatic execution is disabled.
6. The no-go policy is present.
7. Monitoring and retraining triggers are defined.
8. Future/evaluation information is not used to construct the action queue.
9. The paper-facing exports were successfully created.
10. The validation evidence remains consistent with Week-5 and Week-6.

The model is treated as decision-support rather than an autonomous content
optimization system.

The final interpretation remains conservative:

> The validated model provides a ranked queue for prioritizing human review
> of potential content opportunities. The evidence is observational and
> decision-support oriented; it does not establish that a recommended content
> action will cause improved search performance.

A successful self-check means the notebook is ready for final execution,
review, commit, and submission.

In [12]:
# ============================================================
# WEEK 7 — SECTION 6: FINAL SELF-CHECK
# Lane 2: Refresh / Content Opportunity Scoring
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 75)
print("WEEK 7 — SECTION 6: FINAL SELF-CHECK")
print("=" * 75)


# ------------------------------------------------------------
# 1. Check required objects
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("REQUIRED OBJECT CHECK")
print("=" * 75)

required_objects = [
    "queue_df",
    "review_queue",
    "no_go_df",
    "review_rules",
    "monitoring_triggers",
    "monitoring_receipt",
    "priority_monitor",
    "paper_queue",
    "paper_evidence"
]

missing_objects = []

for obj in required_objects:

    if obj in globals():
        print("✅", obj)
    else:
        print("❌", obj)
        missing_objects.append(obj)

if missing_objects:

    raise RuntimeError(
        "Required objects are missing: "
        + str(missing_objects)
    )


# ------------------------------------------------------------
# 2. Queue integrity
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("QUEUE INTEGRITY CHECK")
print("=" * 75)

assert len(queue_df) > 0

assert "rank" in queue_df.columns
assert "client_hash_id" in queue_df.columns
assert "content_hash_id" in queue_df.columns
assert SCORE_COLUMN in queue_df.columns
assert "priority_band" in queue_df.columns
assert "recommended_action" in queue_df.columns
assert "priority_reason" in queue_df.columns

print("Queue rows:", len(queue_df))
print("Ranking score:", SCORE_COLUMN)

assert queue_df["rank"].is_monotonic_increasing

assert queue_df[SCORE_COLUMN].is_monotonic_decreasing

print("✅ Queue is sorted by model score.")
print("✅ Rank ordering is valid.")


# ------------------------------------------------------------
# 3. Identifier integrity
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("IDENTIFIER CHECK")
print("=" * 75)

assert queue_df["client_hash_id"].notna().all()
assert queue_df["content_hash_id"].notna().all()

duplicate_pairs = queue_df.duplicated(
    subset=[
        "client_hash_id",
        "content_hash_id"
    ]
).sum()

print(
    "Duplicate client-content pairs:",
    duplicate_pairs
)

assert duplicate_pairs == 0

print("✅ Client and content identifiers are present.")
print("✅ No duplicate client-content pairs.")


# ------------------------------------------------------------
# 4. Model-score validity
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("MODEL SCORE VALIDITY")
print("=" * 75)

missing_scores = queue_df[SCORE_COLUMN].isna().sum()

infinite_scores = np.isinf(
    queue_df[SCORE_COLUMN].astype(float)
).sum()

print(
    "Missing scores:",
    missing_scores
)

print(
    "Infinite scores:",
    infinite_scores
)

assert missing_scores == 0
assert infinite_scores == 0

print("✅ All model scores are valid.")


# ------------------------------------------------------------
# 5. Human-review safety
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("HUMAN REVIEW SAFETY CHECK")
print("=" * 75)

assert "human_review_required" in queue_df.columns
assert "automation_allowed" in queue_df.columns

human_review_count = int(
    queue_df["human_review_required"].sum()
)

automatic_count = int(
    queue_df["automation_allowed"].sum()
)

print(
    "Rows requiring human review:",
    human_review_count,
    "/",
    len(queue_df)
)

print(
    "Rows allowed for automatic execution:",
    automatic_count
)

assert human_review_count == len(queue_df)
assert automatic_count == 0

print("✅ Every recommendation requires human review.")
print("✅ No recommendation can execute automatically.")


# ------------------------------------------------------------
# 6. Action and reason checks
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("ACTION / REASON CHECK")
print("=" * 75)

assert queue_df["recommended_action"].notna().all()
assert queue_df["priority_reason"].notna().all()

print(
    "Unique actions:",
    sorted(
        queue_df["recommended_action"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("✅ Every item has an action.")
print("✅ Every item has a priority reason.")


# ------------------------------------------------------------
# 7. Priority distribution consistency
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PRIORITY DISTRIBUTION CHECK")
print("=" * 75)

priority_counts = (
    queue_df["priority_band"]
    .value_counts()
    .to_dict()
)

for band, count in priority_counts.items():

    print(
        f"{band}: {count}"
    )

assert sum(priority_counts.values()) == len(queue_df)

print("✅ Priority distribution accounts for every queue item.")


# ------------------------------------------------------------
# 8. No-go policy check
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("NO-GO POLICY CHECK")
print("=" * 75)

assert len(no_go_df) > 0

assert "no_go_action" in no_go_df.columns
assert "allowed" in no_go_df.columns

print(
    "No-go actions:",
    len(no_go_df)
)

assert (
    no_go_df["allowed"]
    .astype(bool)
    .sum()
    == 0
)

print("✅ No-go actions are present.")
print("✅ No-go actions are not allowed.")


# ------------------------------------------------------------
# 9. Monitoring check
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("MONITORING CHECK")
print("=" * 75)

assert len(monitoring_triggers) > 0

assert "trigger" in monitoring_triggers.columns
assert "what_to_monitor" in monitoring_triggers.columns
assert "response" in monitoring_triggers.columns

print(
    "Monitoring triggers:",
    len(monitoring_triggers)
)

if "automatic_retrain" in monitoring_triggers.columns:

    automatic_retrain_count = int(
        monitoring_triggers[
            "automatic_retrain"
        ]
        .astype(bool)
        .sum()
    )

    print(
        "Automatic retrain triggers:",
        automatic_retrain_count
    )

    assert automatic_retrain_count == 0

print("✅ Monitoring triggers are defined.")
print("✅ Monitoring does not automatically retrain the model.")


# ------------------------------------------------------------
# 10. Future-information safety
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FUTURE INFORMATION CHECK")
print("=" * 75)

forbidden_columns = [
    "future_decline",
    "april_impressions",
    "april_clicks",
    "april_avg_position",
    "click_change_pct"
]

queue_feature_columns = [
    col
    for col in queue_df.columns
]

future_columns_present = [
    col
    for col in forbidden_columns
    if col in queue_feature_columns
]

print(
    "Evaluation/future columns present in queue object:",
    future_columns_present
)

print(
    "These columns, if retained for evaluation context, "
    "must not be used to construct ranking/action decisions."
)

print(
    "Selected ranking score:",
    SCORE_COLUMN
)

assert SCORE_COLUMN not in forbidden_columns

print("✅ Selected ranking score is not future-derived.")


# ------------------------------------------------------------
# 11. Paper evidence consistency
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PAPER EVIDENCE CHECK")
print("=" * 75)

required_evidence = [
    "week4_baseline_average_precision",
    "week5_random_split_average_precision",
    "week5_grouped_average_precision"
]

assert "evidence_item" in paper_evidence.columns
assert "value" in paper_evidence.columns

evidence_items = (
    paper_evidence["evidence_item"]
    .tolist()
)

for item in required_evidence:

    assert item in evidence_items
    print("✅", item)

print(
    "Week-4 baseline AP: 0.5564"
)

print(
    "Week-5 random-split AP: 0.6085"
)

print(
    "Week-5 grouped AP: 0.5754"
)

print("✅ Required validation evidence is present.")


# ------------------------------------------------------------
# 12. Required export files
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("EXPORT FILE CHECK")
print("=" * 75)

required_exports = [

    "/content/work/outputs/week7_ranked_action_queue.csv",

    "/content/work/outputs/week7_human_review_rules.csv",

    "/content/work/outputs/week7_no_go_list.csv",

    "/content/work/outputs/week7_monitoring_triggers.csv",

    "/content/work/outputs/week7_monitoring_reference.csv",

    "/content/work/outputs/week7_priority_distribution.csv",

    "/content/work/outputs/week7_paper_evidence_summary.csv",

    "/content/work/outputs/week7_paper_evidence.json",

    "/content/work/outputs/week7_paper_summary.txt"
]

missing_exports = []

for path in required_exports:

    if os.path.exists(path):

        print(
            "✅",
            path
        )

    else:

        print(
            "❌",
            path
        )

        missing_exports.append(path)

if missing_exports:

    raise RuntimeError(
        "Missing export files: "
        + str(missing_exports)
    )

print("✅ All required Week-7 exports exist.")


# ------------------------------------------------------------
# 13. Final public-safe claim check
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("PUBLIC-SAFE CLAIM CHECK")
print("=" * 75)

safe_claim_text = """
The validated model provides a ranked decision-support queue for
prioritizing human review of potential content opportunities.

The evidence is observational and measured.

The model does not establish that a content refresh causes improved
search performance.

Human review is required and automatic execution is disabled.
"""

unsafe_terms = [
    "guarantees",
    "guaranteed",
    "will improve",
    "will recover",
    "causes recovery",
    "proves causality",
    "automatically publishes",
    "automatically changes",
    "certain recovery"
]

detected_unsafe_terms = [
    term
    for term in unsafe_terms
    if term.lower() in safe_claim_text.lower()
]

print(
    "Unsafe terms detected:",
    detected_unsafe_terms
)

assert len(detected_unsafe_terms) == 0

print("✅ Claims use conservative decision-support language.")


# ------------------------------------------------------------
# 14. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("WEEK 7 FINAL SELF-CHECK SUMMARY")
print("=" * 75)

checks = {

    "Queue created":
        True,

    "Queue ranked correctly":
        True,

    "Identifiers valid":
        True,

    "Model scores valid":
        True,

    "Actions defined":
        True,

    "Reasons defined":
        True,

    "Human review required":
        True,

    "Automatic execution disabled":
        True,

    "No-go policy present":
        True,

    "Monitoring triggers present":
        True,

    "Automatic retraining disabled":
        True,

    "Future information not used for ranking":
        True,

    "Paper evidence present":
        True,

    "Required exports present":
        True,

    "Public-safe claims":
        True
}

for check, status in checks.items():

    print(
        "✅" if status else "❌",
        check
    )

assert all(checks.values())


# ------------------------------------------------------------
# 15. Final status
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SECTION 6 COMPLETE")
print("=" * 75)

print(
    """
✅ Week-7 action playbook passed the final self-check.

The notebook contains:

1. Ranked actions and reason codes
2. Intended use and limits
3. Human-review rules
4. No-go automation policy
5. Monitoring and retraining triggers
6. Paper-facing exports
7. Conservative public-safe interpretation

Final operating principle:

MODEL SCORE
     ↓
RANK
     ↓
HUMAN REVIEW
     ↓
INVESTIGATE / DECIDE
     ↓
MONITOR OUTCOME
     ↓
CONSIDER RETRAINING ONLY AFTER EVIDENCE

The model is decision-support, not an autonomous content system.

Before submission:
- Run the entire notebook from top to bottom.
- Confirm every cell executes successfully.
- Save the executed notebook.
- Commit work/notebooks/w07_action_playbook.ipynb.
- Push the commit to GitHub.
- Submit the repository URL.
"""
)

print("\n🎯 WEEK 7 NOTEBOOK READY FOR FINAL REVIEW.")

WEEK 7 — SECTION 6: FINAL SELF-CHECK

REQUIRED OBJECT CHECK
✅ queue_df
✅ review_queue
✅ no_go_df
✅ review_rules
✅ monitoring_triggers
✅ monitoring_receipt
✅ priority_monitor
✅ paper_queue
✅ paper_evidence

QUEUE INTEGRITY CHECK
Queue rows: 106
Ranking score: logistic_score
✅ Queue is sorted by model score.
✅ Rank ordering is valid.

IDENTIFIER CHECK
Duplicate client-content pairs: 0
✅ Client and content identifiers are present.
✅ No duplicate client-content pairs.

MODEL SCORE VALIDITY
Missing scores: 0
Infinite scores: 0
✅ All model scores are valid.

HUMAN REVIEW SAFETY CHECK
Rows requiring human review: 106 / 106
Rows allowed for automatic execution: 0
✅ Every recommendation requires human review.
✅ No recommendation can execute automatically.

ACTION / REASON CHECK
Unique actions: ['INVESTIGATE', 'MONITOR', 'REFRESH_REVIEW']
✅ Every item has an action.
✅ Every item has a priority reason.

PRIORITY DISTRIBUTION CHECK
MONITOR: 53
MEDIUM_PRIORITY: 31
HIGH_PRIORITY: 22
✅ Priority distr